# Import Required Libraries
Import necessary libraries including NLTK, transformers (for LLMs), scikit-learn (for ML models), and other utility packages.

In [1]:
# Import necessary libraries
import nltk  # For working with the NLTK treebank dataset
from transformers import pipeline  # For working with large language models (LLMs)
from sklearn.model_selection import train_test_split  # For splitting datasets
from sklearn.metrics import classification_report  # For evaluating ML models
import numpy as np  # For numerical operations
import pandas as pd  # For data manipulation and analysis
import torch

# Download the NLTK treebank dataset
nltk.download('treebank')

/Users/larsheijnen/VSCode_S2/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[nltk_data] Downloading package treebank to
[nltk_data]     /Users/larsheijnen/nltk_data...
[nltk_data]   Package treebank is already up-to-date!


True

# Load and Prepare NLTK Treebank Dataset
Download and prepare the NLTK treebank dataset, including preprocessing steps specific to probing tasks.

In [2]:
# Load and Prepare NLTK Treebank Dataset

# Load the NLTK treebank dataset
from nltk.corpus import treebank

# Extract sentences and their corresponding part-of-speech (POS) tags
sentences = treebank.sents()
pos_tags = treebank.tagged_sents()

# Convert sentences and POS tags into a DataFrame for easier manipulation
data = pd.DataFrame({
    'sentence': [' '.join(sentence) for sentence in sentences],
    'pos_tags': [' '.join(f"{word}/{tag}" for word, tag in tagged_sentence) for tagged_sentence in pos_tags]
})

# Split the dataset into training and testing sets
train_data, test_data = train_test_split(data, test_size=0.2, random_state=42)

# Display the first few rows of the training data
train_data.head()

,sentence,pos_tags
0,"Pierre Vinken , 61 years old , will join the b...","Pierre/NNP Vinken/NNP ,/, 61/CD years/NNS old/..."
2117,And it was stupid .,And/CC it/PRP was/VBD stupid/JJ ./.
1091,Some dealers said 0 the dollar was pressured *...,Some/DT dealers/NNS said/VBD 0/-NONE- the/DT d...
3763,The new company will attempt *-1 to limit the ...,The/DT new/JJ company/NN will/MD attempt/VB *-...
2282,The framers hardly discussed the appropriation...,The/DT framers/NNS hardly/RB discussed/VBD the...


# Define Pipeline Architecture
Create abstract base classes and interfaces for the modular pipeline components, ensuring they can be interchanged easily.

In [3]:
from abc import ABC, abstractmethod

# Define an abstract base class for pipeline components
class PipelineComponent(ABC):
    @abstractmethod
    def process(self, input_data):
        """
        Process the input data and return the output data.
        This method must be implemented by all subclasses.
        """
        pass

# Define an abstract base class for the pipeline itself
class ModularPipeline(ABC):
    def __init__(self):
        self.components = []  # List to hold pipeline components

    def add_component(self, component):
        """
        Add a component to the pipeline.
        """
        if not isinstance(component, PipelineComponent):
            raise TypeError("Component must be an instance of PipelineComponent")
        self.components.append(component)

    def remove_component(self, component):
        """
        Remove a component from the pipeline.
        """
        self.components.remove(component)

    def execute(self, input_data):
        """
        Execute the pipeline by passing the input data through all components.
        """
        data = input_data
        for component in self.components:
            data = component.process(data)
        return data

# Example of a concrete implementation of a pipeline component
class TokenizerComponent(PipelineComponent):
    def process(self, input_data):
        """
        Tokenize the input sentences.
        """
        return [sentence.split() for sentence in input_data]

# Example of a concrete implementation of a pipeline component
class POSTaggerComponent(PipelineComponent):
    def process(self, input_data):
        """
        Perform POS tagging on tokenized sentences.
        """
        return [nltk.pos_tag(sentence) for sentence in input_data]

# Create Feature Extraction Module
Implement functions to extract embeddings and other features from LLMs, with different strategies that can be plugged into the pipeline.

In [4]:
# Define a feature extraction module as a pipeline component
class FeatureExtractionComponent(PipelineComponent):
    def __init__(self, model_name="bert-base-uncased", layer=-1, strategy="mean"):
        """
        Initialize the feature extraction component.
        
        Parameters:
        - model_name: Name of the pre-trained model to use for embeddings.
        - layer: The specific layer of the model to extract embeddings from.
        - strategy: Strategy to aggregate token embeddings ('mean', 'max', or 'cls').
        """
        from transformers import AutoModel, AutoTokenizer
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name, output_hidden_states=True)
        self.layer = layer
        self.strategy = strategy

    def process(self, input_data):
        """
        Extract features (embeddings) from input sentences.
        
        Parameters:
        - input_data: List of sentences to process.
        
        Returns:
        - List of extracted embeddings for each sentence.
        """
        embeddings = []
        for sentence in input_data:
            # Tokenize the sentence
            inputs = self.tokenizer(sentence, return_tensors="pt", truncation=True, padding=True)
            
            # Get model outputs
            with torch.no_grad():
                outputs = self.model(**inputs)
            
            # Extract hidden states
            hidden_states = outputs.hidden_states[self.layer]
            
            # Aggregate embeddings based on the strategy
            if self.strategy == "mean":
                sentence_embedding = hidden_states.mean(dim=1).squeeze().numpy()
            elif self.strategy == "max":
                sentence_embedding = hidden_states.max(dim=1).values.squeeze().numpy()
            elif self.strategy == "cls":
                sentence_embedding = hidden_states[:, 0, :].squeeze().numpy()
            else:
                raise ValueError("Invalid strategy. Choose from 'mean', 'max', or 'cls'.")
            
            embeddings.append(sentence_embedding)
        
        return embeddings

# Example usage of the FeatureExtractionComponent
# Initialize the feature extraction component
feature_extractor = FeatureExtractionComponent(model_name="bert-base-uncased", layer=-1, strategy="mean")

# Add the feature extraction component to the pipeline
pipeline = ModularPipeline()
pipeline.add_component(feature_extractor)

# Example input sentences
example_sentences = train_data['sentence'].tolist()[:5]

# Execute the pipeline to extract features
extracted_features = pipeline.execute(example_sentences)

# Display the shape of the extracted features for the first sentence
print(f"Shape of extracted features for the first sentence: {np.array(extracted_features[0]).shape}")

Shape of extracted features for the first sentence: (768,)
